In [19]:
from langchain_ollama import ChatOllama 

In [20]:
llm=ChatOllama(model="llama3.1:latest")

In [21]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [22]:
@tool 
def multiply(a:int,b:int)->int:
    """Multiply 2 given numbers"""
    return a*b

In [23]:
print(multiply.invoke({"a":4,"b":3}))

12


In [24]:
multiply.description

'Multiply 2 given numbers'

**TOOL BINDING**

In [25]:
llm_with_tools= llm.bind_tools([multiply])

In [ ]:
# llm_with_tools.invoke('Hi How are you?')

AIMessage(content='I didn\'t receive a mathematical question that requires calling one of the provided functions. A simple response would be:\n"I\'m doing well, thank you for asking!" \n\nHowever, if I had to provide an answer in the requested format with a JSON object and a function call that doesn\'t apply to this prompt:\n\n{"name": "multiply", "parameters": {}}', additional_kwargs={}, response_metadata={'model': 'llama3.1:latest', 'created_at': '2026-08-29T17:05:36.584622851Z', 'done': True, 'done_reason': 'stop', 'total_duration': 30410339762, 'load_duration': 13291853868, 'prompt_eval_count': 168, 'prompt_eval_duration': 7510875028, 'eval_count': 74, 'eval_duration': 9600608520, 'logprobs': None, 'model_name': 'llama3.1:latest', 'model_provider': 'ollama'}, id='lc_run--01a04e7b-6a7d-75c1-b046-d033552e717d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 168, 'output_tokens': 74, 'total_tokens': 242})

In [31]:
result=llm_with_tools.invoke('multiply 20 with 50') # does not run the tool it just suggest the tool

In [33]:
result.tool_calls[0]  # SUGGESTION FROM LLM THAT YOU SHOULD CALL MULTIPLY FUNCTION

{'name': 'multiply',
 'args': {'a': 20, 'b': 50},
 'id': 'ba100ec8-3dd7-4668-989f-7d707cdb317a',
 'type': 'tool_call'}

In [35]:
multiply.invoke({'a': 20, 'b': 50})

1000

In [34]:
multiply.invoke(result.tool_calls[0]['args'])

1000

In [37]:
# gives output as tool message

multiply.invoke({'name': 'multiply',
 'args': {'a': 20, 'b': 50},
 'id': 'ba100ec8-3dd7-4668-989f-7d707cdb317a',
 'type': 'tool_call'})

ToolMessage(content='1000', name='multiply', tool_call_id='ba100ec8-3dd7-4668-989f-7d707cdb317a')

In [ ]:
multiply.invoke(result.tool_calls[0])  # same as earlier one

ToolMessage(content='1000', name='multiply', tool_call_id='ba100ec8-3dd7-4668-989f-7d707cdb317a')

**WE MAINTAIN A HISTORY OF MESSAGES RATHER THAN JUST CALLING IT EVERY STEP**

In [40]:
query=HumanMessage("multiplt 10 with 59")

In [41]:
messages=[query]   # maintain the history

In [44]:
messages

[HumanMessage(content='multiplt 10 with 59', additional_kwargs={}, response_metadata={})]

In [53]:
llm_with_tools= llm.bind_tools([multiply])

In [43]:
result1=llm_with_tools.invoke(messages)

In [46]:
messages.append(result1)  # store the result in message i.e the AI Message

In [47]:
messages

[HumanMessage(content='multiplt 10 with 59', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.1:latest', 'created_at': '2026-08-29T17:18:57.703083583Z', 'done': True, 'done_reason': 'stop', 'total_duration': 18333465873, 'load_duration': 7859783677, 'prompt_eval_count': 170, 'prompt_eval_duration': 7721666117, 'eval_count': 22, 'eval_duration': 2751419411, 'logprobs': None, 'model_name': 'llama3.1:latest', 'model_provider': 'ollama'}, id='lc_run--01a04e87-d308-7620-9715-ce82f6581647-0', tool_calls=[{'name': 'multiply', 'args': {'a': 10, 'b': 59}, 'id': '487e3a56-f61a-4661-a6ce-78e444efbeb6', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 170, 'output_tokens': 22, 'total_tokens': 192})]

In [48]:
tool_result=multiply.invoke(result1.tool_calls[0])

In [49]:
tool_result

ToolMessage(content='590', name='multiply', tool_call_id='487e3a56-f61a-4661-a6ce-78e444efbeb6')

In [50]:
messages.append(tool_result)

In [52]:
messages  # contains all the messages human, AI and tool

[HumanMessage(content='multiplt 10 with 59', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.1:latest', 'created_at': '2026-08-29T17:18:57.703083583Z', 'done': True, 'done_reason': 'stop', 'total_duration': 18333465873, 'load_duration': 7859783677, 'prompt_eval_count': 170, 'prompt_eval_duration': 7721666117, 'eval_count': 22, 'eval_duration': 2751419411, 'logprobs': None, 'model_name': 'llama3.1:latest', 'model_provider': 'ollama'}, id='lc_run--01a04e87-d308-7620-9715-ce82f6581647-0', tool_calls=[{'name': 'multiply', 'args': {'a': 10, 'b': 59}, 'id': '487e3a56-f61a-4661-a6ce-78e444efbeb6', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 170, 'output_tokens': 22, 'total_tokens': 192}),
 ToolMessage(content='590', name='multiply', tool_call_id='487e3a56-f61a-4661-a6ce-78e444efbeb6')]